In [1]:
import pandas as pd
import hashlib
import csv

In [2]:
Df_training_set = pd.read_csv('FrontendMavenPlugin.csv', sep='$')
Df_training_set.dropna()
Df_training_set.columns = [col.strip() for col in Df_training_set.columns]
len(Df_training_set)

4994

In [3]:
Df_training_set

,id,,,,.,a,if,return,throw,void,...,"} """,} ',} :,} <,~,~ /,~ dp0node,class_name,class_value,contents
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,build_status,1,++ b/README.md
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,build_status,1,"1,23 @@"
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,build_status,1,This plugin downloads/installs Node and NPM lo...
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,build_status,1,"It's supposed to work on Windows, OS X and Linux."
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,build_status,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4989,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,build_status,0,++ b/frontend-maven-plugin/src/main/java/com/g...
4990,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,build_status,0,}
4991,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,build_status,0,"39,6 @@ public final class BowerMojo extends A..."
4992,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,build_status,0,this.additionalEnvironment = additiona...


In [4]:
Df_training_set = Df_training_set.loc[:,~Df_training_set.columns.duplicated()].copy()
Df_training_set

,id,,.,a,if,return,throw,void,},!,...,| |,"} """,} ',} :,} <,~ /,~ dp0node,class_name,class_value,contents
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,build_status,1,++ b/README.md
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,build_status,1,"1,23 @@"
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,build_status,1,This plugin downloads/installs Node and NPM lo...
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,build_status,1,"It's supposed to work on Windows, OS X and Linux."
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,build_status,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4989,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,build_status,0,++ b/frontend-maven-plugin/src/main/java/com/g...
4990,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,build_status,0,}
4991,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,build_status,0,"39,6 @@ public final class BowerMojo extends A..."
4992,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,build_status,0,this.additionalEnvironment = additiona...


In [ ]:
# d["result"][-1:]

In [ ]:
# for col in Df_training_set.columns:
#     print(col)
# for index, row in Df_training_set.iterrows():
#     print(row[col])

In [5]:
class CLineInfo:
    def __init__(self):
        iHash = 0
        iCounter= 0
        strFileName = ""
        strCodeLine = ""
        strClassName=""
        strClassValue= ""
        strTestName = ""
        strPth=""
        iVerdict = 0

In [6]:
lineDictionary = {}

In [7]:
def addHashToLine():    
    for i, strInputLine in Df_training_set.iterrows(): 
        strCodeLine = str(strInputLine['contents'])
        iHash = int(hashlib.sha1(strCodeLine.encode('utf-8')).hexdigest(), 16) % (10 ** 8)
        # Df_training_set.set_value(i,'contents',strCodeLine)
        # Df_training_set.set_value(i,'hash',iHash)
        Df_training_set.at[i, 'contents_x'] = strCodeLine
        Df_training_set.at[i, 'hasheddd'] = iHash

In [8]:
def createDictionary():
    iLineIndex = 0
    for index, strInputLine in Df_training_set.iterrows():  
            if iLineIndex != 0:            
                lineObject = CLineInfo()
                lineObject.iHash = int(strInputLine['hasheddd'])
                lineObject.strFileName = strInputLine['id']
                lineObject.strCodeLine = strInputLine['contents_x']
                lineObject.iVerdict = strInputLine['result']
                lineObject.strClassName = strInputLine['class_name']
                lineObject.strClassValue = strInputLine['class_value']
                lineObject.strPath = strInputLine['path']
                lineObject.strTestName = strInputLine['test']
                lineObject.iCounter = strInputLine['line']
                # check if the line already exists in the dictionary
                if lineObject.iHash in lineDictionary:
                    old_line_verdict = lineDictionary[lineObject.iHash].iVerdict
                    # check if the new occurrence has a different verdict and that lineObject has a verdict of 1, 
                    # if so, then we relabel the dictionary line by replacing it with the lineObject
                    if (old_line_verdict != lineObject.iVerdict) and (lineObject.iVerdict == 1.0):
                          lineDictionary[lineObject.iHash] = lineObject
                    else:
                        continue #do nothing and keep the lineObject as is in the dictionary
                else:                
                    lineDictionary[lineObject.iHash] = lineObject
            iLineIndex += 1

In [9]:
addHashToLine()
createDictionary()
len(lineDictionary)

1559

In [10]:
outputFile = "./code_test_no_noise.csv"
fOutputFile = open(outputFile, 'w', encoding = 'utf-8')

In [11]:
with open('./code_test_no_noise.csv', mode='w') as csv_file:
    header_names = ['id', 'line', 'contents', 'class_name', 'class_value', 'path' , 'testName', 'verdict'] #, 'verdict', 'hash'
    writer = csv.DictWriter(csv_file, fieldnames=header_names, quotechar='"', 
                            quoting=csv.QUOTE_ALL, delimiter=',')
    writer.writeheader()
    for hashKey, oneLineObject in lineDictionary.items():
        writer.writerow({'id': oneLineObject.strFileName, 
                         'line': str(oneLineObject.iCounter), 
                         'contents': oneLineObject.strCodeLine, 
                         'class_name': oneLineObject.strClassName, 
                         'class_value': oneLineObject.strClassValue,
                         'testName' : oneLineObject.strTestName,
                         'path': oneLineObject.strPath,
                         'verdict': oneLineObject.iVerdict }) 
        #, 'verdict': str(oneLineObject.iVerdict),'hash': oneLineObject.iHash
fOutputFile.close()

In [12]:
no_noise = pd.read_csv('code_test_no_noise.csv', sep=',', encoding = 'ISO-8859-1')
no_noise

,id,line,contents,class_name,class_value,path,testName,verdict
0,0,0,"1,23 @@",build_status,1,0,0,0
1,0,0,This plugin downloads/installs Node and NPM lo...,build_status,1,0,0,0
2,0,0,"It's supposed to work on Windows, OS X and Linux.",build_status,1,0,0,0
3,0,0,NaN,build_status,1,0,0,0
4,0,0,# Installing,build_status,1,0,0,0
...,...,...,...,...,...,...,...,...
1554,0,0,"12,7 @@ import java.util.List;",build_status,0,0,0,0
1555,0,0,factory.getJspmRunner().execute(argume...,build_status,0,0,0,0
1556,0,0,"public final void execute(String args, Map...",build_status,0,0,0,0
1557,0,0,"61,7 @@ public final class EmberMojo extends A...",build_status,0,0,0,0
